# step1 获取数据，拆解token

In [ ]:
import torch

from pathlib import Path
from urllib.request import urlretrieve

data_path = Path.cwd() / 'a2a_shakesphere.txt'
url = (
    "https://raw.githubusercontent.com/"
    "karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
)

if not data_path.exists():
    urlretrieve(url,data_path)

text = data_path.read_text(encoding= 'utf-8').lower()
print(text[:100])



In [ ]:
chars = sorted(set(text))
print(chars)

chars_to_ids = {c:i for i,c in enumerate(chars)}
ids_to_chars = {i:c for i,c in enumerate(chars)}

mask_id = len(chars)
vocab_size = len(chars)+1

print(chars_to_ids)

In [ ]:
data = torch.tensor([chars_to_ids[c] for c in text],dtype=torch.long)
print(data[:100])
split = int(len(data)*0.9)
train_data = data[:split]
test_data = data[split:]


# step2 batch sampling

In [ ]:
torch.manual_seed(0)
def get_batch(data,prefix_len,block_size,batch_size):

    seq_len = prefix_len+block_size

    starts = torch.randint(0,len(data)-seq_len+1,(batch_size,))


    sample = torch.stack(
        [data[start.item():start.item()+seq_len] for start in starts]
        )

    input_ids = sample.clone()

    # 部分遮挡 50%
    mask_positions = torch.rand(batch_size, block_size) < 0.5
    mask_positions[~mask_positions.any(dim=1), 0] = True # 至少把第一个设置成 True
    input_ids[:,prefix_len:]= input_ids[:,prefix_len:].masked_fill(mask_positions,mask_id)

    # 25% 几率全遮挡
    full_mask_rows = torch.rand(batch_size)< 0.25
    input_ids[full_mask_rows,prefix_len:] = mask_id
    mask_positions[full_mask_rows, :] = True

    target = sample[:,prefix_len:].clone()
    return sample, input_ids, target,mask_positions  # batch * sequence len

prefix_len,block_size,batch_size = 64,16,4
sample,input_ids,target,mask_positions = get_batch(train_data,64,16,4)

print(input_ids)

# step 3 embedding

In [ ]:
d_model = 64
max_seq_len = 128

token_embedding = torch.nn.Embedding(vocab_size,d_model) # 37*64
position_embedding = torch.nn.Embedding(max_seq_len,d_model)
position_ids = torch.stack([torch.arange(input_ids.shape[-1]) for i in range(input_ids.shape[0])])

x = token_embedding(input_ids)+position_embedding(position_ids)
print(x.shape)


# step4 forward attention

In [ ]:
class Attention(torch.nn.Module):
    def __init__(self,d_model):
        super().__init__()

        self.w_q = torch.nn.Parameter(
            torch.randn(d_model, d_model) * d_model**-0.5
        )
        self.w_k = torch.nn.Parameter(
            torch.randn(d_model, d_model) * d_model**-0.5
        )
        self.w_v = torch.nn.Parameter(
            torch.randn(d_model, d_model) * d_model**-0.5
        )

        self.out_proj = torch.nn.Linear(d_model, d_model) #线性层
        self.norm = torch.nn.LayerNorm(d_model) #归一和线性层
        self.norm2 = torch.nn.LayerNorm(d_model)

        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(d_model, 4 * d_model),
            torch.nn.GELU(),
            torch.nn.Linear(4 * d_model, d_model)
        )#mlp 层

    def forward(self,x,prefix_len):
        original = x
        x = self.norm(x)
        q = x@self.w_q #[4*80*64]
        k = x@self.w_k
        v = x@self.w_v

        score = q @ k.transpose(-2,-1) / q.shape[-1] ** 0.5 #[4*80*80]
        # AR mask
        allowed = torch.ones(q.shape[-2],q.shape[-2],dtype=torch.bool,device=x.device).tril()
        # BD mask
        allowed[prefix_len:, :] = True # 最后几个token 全部可以看 [80*80]

        scores = score.masked_fill(~allowed, float("-inf"))
        weights = scores.softmax(dim=-1)
        context = weights @ v #[4,80,64]
        output = self.out_proj(context)
        h = output + original
        h = h + self.mlp(self.norm2(h))
        return h

    
attention = Attention(64)
attention.forward(x,prefix_len=prefix_len)


# step5 正式bd

In [ ]:
class ToyBD(torch.nn.Module):
    def __init__(self,vocab_size,d_model,max_seq_len,):
        super().__init__()
        self.token_embedding = torch.nn.Embedding(vocab_size,d_model)
        self.position_embedding = torch.nn.Embedding(max_seq_len,d_model)

        self.attention = Attention(d_model)
        self.final_norm = torch.nn.LayerNorm(d_model)
        self.output_head = torch.nn.Linear(d_model, vocab_size)

    def forward(self, input_ids, prefix_len):
        seq_len = input_ids.shape[1] #[batchsize,len]
        positions = torch.arange(seq_len, device=input_ids.device)

        x = (
            self.token_embedding(input_ids)
            + self.position_embedding(positions)
        )#[batchsize,len,d_model]

        h = self.attention(x, prefix_len)#[batchsize,len,d_model]
        logits = self.output_head(self.final_norm(h))#[batchsize,len,vocab_size]

        return logits

# Step 6 开始训练

In [ ]:
prefix_len = 128
block_size = 4
batch_size = 100

vocab_size = vocab_size
d_model = 128
max_seq_len = 256

num_steps = 3000


bd_model = ToyBD(vocab_size,d_model=d_model, max_seq_len=max_seq_len)
optimizer = torch.optim.AdamW(bd_model.parameters(),lr = 1e-3)

loss_record = []
for i in range(num_steps):
    sample ,input_ids, targets,mask_positions = get_batch(train_data,prefix_len,block_size,batch_size)
    optimizer.zero_grad()
    logits = bd_model.forward(input_ids=input_ids,prefix_len=prefix_len)
    block_logits = logits[:,prefix_len:,:]
    loss = torch.nn.functional.cross_entropy(block_logits[mask_positions],targets[mask_positions]) # [4, 16, vocablen] and # [4, 16] reshape: 自动合并成目标维度
    # 取出后自动变成一维
    loss.backward()
    loss_record.append(loss.item())
    print('step',i,'loss',loss.item())
    optimizer.step()
 
print(loss_record)
import matplotlib.pyplot as plt
plt.plot(loss_record)






# step7 rollout

In [ ]:
bd_model.eval()  # 切换到推理模式

# 从验证集取64个字符作为开头，不输入后面的真实答案
generated = test_data[1000:1000+prefix_len].clone()
device = next(bd_model.parameters()).device
generated = generated.to(device)

print("前文：")
print("".join(ids_to_chars[i] for i in generated.tolist()))

with torch.no_grad():  # 推理时不计算梯度
    for _ in range(5):  # 生成5块，每块16个字符
        # 始终只保留最近64个字符作为模型的上下文
        prefix = generated[-prefix_len:]

        masks = torch.full(
            (block_size,), mask_id,
            dtype=torch.long, device=device
        )

        # [80] → [1, 80]，增加batch维度
        inputs = torch.cat([prefix, masks]).unsqueeze(0)

        for denoise_step in range(block_size):
            logits = bd_model.forward(inputs,prefix_len)
            block_logits = logits[0,prefix_len:].clone()
            block_logits[:, mask_id] = float('-inf')
            prob = block_logits.softmax(dim = -1)

            confidence, predicted_ids = prob.max(dim=-1)

            # 已填好的位置不再参与选择
            unknown = inputs[0, prefix_len:] == mask_id
            confidence = confidence.masked_fill(~unknown, -1.0)

            position = confidence.argmax().item()
            inputs[0, prefix_len+position] = predicted_ids[position]

        generated = torch.cat([generated, inputs[0,prefix_len:]])




print("\n完整结果：")
print("".join(ids_to_chars[i] for i in generated.tolist()))